In [ ]:
# check available CUDA devices

import torch

devices = []

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        device_name = torch.cuda.get_device_name(i)
        devices.append({
            "type": "CUDA",
            "available": True,
            "name": device_name,
            "index": i
        })
else:
    devices.append({"type": "CUDA", "available": False, "name": "N/A"})

devices

In [ ]:
# change folder into the root of the ASR project

import os

if not os.path.isdir("Configs"):
    %cd ../

!pwd

In [ ]:
# import packages, define common functions

import os
from pathlib import Path
from typing import List, Optional

import numpy as np
import pandas as pd
import torch
import yaml

from meldataset import build_dataloader
from models import ASRCNN
from token_map import build_token_map_from_data


def cfg_get_nested(cfg: dict, path, default=None, sep="."):
    if isinstance(path, str):
        keys = path.split(sep)
    else:
        keys = list(path)

    cur = cfg
    for key in keys:
        if isinstance(cur, dict) and key in cur:
            cur = cur[key]
        else:
            return default
    return cur


def load_ASR_models(ASR_MODEL_PATH: str, ASR_MODEL_CONFIG: str) -> ASRCNN:
    def _load_model(model_config, model_path):
        model = ASRCNN(**model_config)
        params = torch.load(model_path, map_location="cpu", weights_only=False)["model"]
        try:
            model.load_state_dict(params)
        except Exception:
            new_state_dict = {k.replace("module.", ""): v for k, v in params.items()}
            model.load_state_dict(new_state_dict)
        return model

    with open(ASR_MODEL_CONFIG, "r", encoding="utf-8") as f:
        config = yaml.safe_load(f)

    token_src = config.get("phoneme_maps_path")
    if not token_src:
        token_map = build_token_map_from_data(
            config.get("train_data"),
            config.get("val_data"),
            config.get("ood_data"),
            apply_asr_tokenizer=True,
        )
    elif isinstance(token_src, dict):
        token_map = token_src
    else:
        csv = pd.read_csv(token_src, header=None).values
        token_map = {word: index for word, index in csv}

    model_params = cfg_get_nested(
        config,
        "model_params",
        {
            "input_dim": 80,
            "hidden_dim": 256,
            "n_token": len(token_map),
            "token_embedding_dim": 512,
            "n_layers": 5,
            "location_kernel_size": 31,
        },
    )

    if "n_token" not in model_params:
        model_params["n_token"] = len(token_map)

    asr_model = _load_model(model_params, ASR_MODEL_PATH)
    return asr_model


def _load_split_list(list_path: str) -> List[List[str]]:
    entries: List[List[str]] = []
    list_path = Path(list_path)
    if not list_path.exists():
        raise FileNotFoundError(f"Cannot find data list: {list_path}")

    with list_path.open("r", encoding="utf-8") as f:
        for raw_line in f:
            line = raw_line.strip()
            if not line:
                continue
            parts = line.split("|")
            if len(parts) < 3:
                if len(parts) == 2:
                    parts.append("0")
                elif len(parts) == 1:
                    parts.extend(["", "0"])
            path = parts[0]
            if len(parts) > 3:
                text = "|".join(parts[1:-1])
            else:
                text = parts[1]
            speaker_id = parts[-1]
            entries.append([path, text, speaker_id])

    if not entries:
        raise RuntimeError(f"No entries found in {list_path}")
    return entries


def _build_dataset_config(config: dict) -> dict:
    dataset_params = {
        "dict_path": cfg_get_nested(config, "phoneme_maps_path", "Data/word_index_dict.txt"),
        "sr": cfg_get_nested(config, "preprocess_params.sr", 24000),
        "spect_params": cfg_get_nested(
            config,
            "preprocess_params.spect_params",
            {"n_fft": 1024, "win_length": 1024, "hop_length": 300},
        ),
        "mel_params": cfg_get_nested(config, "preprocess_params.mel_params", {"n_mels": 80}),
    }

    dataset_overrides = cfg_get_nested(config, "dataset_params", {})
    if isinstance(dataset_overrides, dict):
        for override_key in ("dict_path", "sr", "spect_params", "mel_params"):
            if override_key in dataset_overrides:
                dataset_params[override_key] = dataset_overrides[override_key]
        if "spec_augment" in dataset_overrides:
            dataset_params["spec_augment_params"] = dataset_overrides["spec_augment"]

    return dataset_params


def create_validation_dataloader(
    config: dict,
    device: str,
    batch_size: Optional[int] = None,
    num_workers: Optional[int] = None,
):
    dataset_config = _build_dataset_config(config)
    val_path = cfg_get_nested(config, "val_data", "Data/val_list.txt")
    val_entries = _load_split_list(val_path)

    if batch_size is None:
        batch_size = min(cfg_get_nested(config, "batch_size", 8), 8)

    dataloader_params = cfg_get_nested(config, "dataloader_params", {})
    if num_workers is None:
        num_workers = int(dataloader_params.get("val_num_workers", 2))

    return build_dataloader(
        val_entries,
        validation=True,
        batch_size=batch_size,
        num_workers=num_workers,
        device=device,
        dataset_config=dataset_config,
    )


def find_latest_checkpoint(checkpoint_dir: Path) -> Path:
    checkpoint_dir = Path(checkpoint_dir)
    if not checkpoint_dir.exists():
        raise FileNotFoundError(f"Checkpoint directory not found: {checkpoint_dir}")

    candidates = []
    for file_path in checkpoint_dir.glob("epoch_*.pth"):
        if not file_path.is_file():
            continue
        stem_parts = file_path.stem.split("_")
        try:
            epoch_id = int(stem_parts[-1])
        except ValueError:
            continue
        candidates.append((epoch_id, file_path))

    if not candidates:
        raise FileNotFoundError(f"No epoch_*.pth checkpoints found in {checkpoint_dir}")

    candidates.sort(key=lambda item: item[0])
    return candidates[-1][1]


def describe_token_source(token_src):
    return token_src if isinstance(token_src, str) else "built from dataset"

In [ ]:
# load the latest ASR model checkpoint and the validation data loader

CONFIG_PATH = Path("Configs/config.yml")
CHECKPOINT_DIR = Path("Checkpoint")
EVAL_BATCH_SIZE = 8

config = yaml.safe_load(CONFIG_PATH.read_text())

model_path = find_latest_checkpoint(CHECKPOINT_DIR)
print(f"model --> {model_path}")
print(f"config --> {CONFIG_PATH}")
print(f"dictionary --> {describe_token_source(config.get('phoneme_maps_path'))}")
print(f"validation data --> {cfg_get_nested(config, 'val_data', 'Data/val_list.txt')}")

device = "cuda" if torch.cuda.is_available() else "cpu"
model = load_ASR_models(str(model_path), str(CONFIG_PATH)).to(device)
model.eval()

val_loader = create_validation_dataloader(config, device=device, batch_size=EVAL_BATCH_SIZE)

print(f"Validation samples: {len(val_loader.dataset)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Using device: {device}")
print("All resources loaded ✓")

In [ ]:
# define the diagonal attention score evaluation

@torch.no_grad()
def diagonal_attention_score(model, dev_loader, device="cuda", band=0.1):
    model_was_training = model.training
    model.eval()

    downsample_factor = 2 ** getattr(model, "n_down", 0)
    scores = []

    for batch in dev_loader:
        texts, text_lens, mels, mel_lens = batch[:4]
        texts = texts.to(device)
        text_lens = text_lens.to(device)
        mels = mels.to(device)
        mel_lens = mel_lens.to(device)

        enc_lens = torch.clamp(mel_lens // downsample_factor, min=1)
        mel_mask = model.length_to_mask(enc_lens).to(device)

        _, _, attn = model(mels, src_key_padding_mask=mel_mask, text_input=texts)

        for idx in range(attn.size(0)):
            tgt_len = int(text_lens[idx].item())
            enc_len = int(enc_lens[idx].item())
            if tgt_len <= 0 or enc_len <= 0:
                continue

            attn_slice = attn[idx, :tgt_len, :enc_len]
            total_mass = attn_slice.sum().clamp_min(1e-8)
            if total_mass.item() == 0.0:
                continue

            to = attn_slice.size(0)
            te = attn_slice.size(1)
            t = torch.arange(to, device=attn_slice.device).unsqueeze(1).float()
            e = torch.arange(te, device=attn_slice.device).unsqueeze(0).float()
            t_norm = t / max(to - 1, 1)
            e_norm = e / max(te - 1, 1)
            diag = t_norm - e_norm
            mask = (diag.abs() <= band).float()
            score = (attn_slice * mask).sum() / total_mass
            scores.append(score.item())

    if model_was_training:
        model.train()

    return float(np.mean(scores)) if scores else float("nan")

In [ ]:
# run the evaluation and report the diagonal attention score

BAND_WIDTH = 0.1

score = diagonal_attention_score(model, val_loader, device=device, band=BAND_WIDTH)
print(f"Diagonal attention score (band={BAND_WIDTH}): {score:.4f}")